## 1. Setup and Cleanup
#### 1.1 Definitions

In [ ]:
from pathlib import Path
import os
import shutil
import json
import re
import pandas as pd
import subprocess
import plotly as pl
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px

#---Data---#
BMS = ["nodeapp","mediawiki","compression","dacapo-spring","dacapo-luindex"]#,"dacapo-lusearch","renaissance-http","renaissance-chirper",
       #"502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000","505.mcf_r.inp","523.xalancbmk_r.xalanc","531.deepsjeng_r.ref","541.leela_r.ref"]

CONF = [2,4,6,8,12,16,24,32,40,50,60,75,100,125,150,175,200]
SAVE = [2,4,6,8,12,16,24,32,40,50,60,75,100]
ALPHA = [0.01,0.05,0.1,0.2,0.3,0.4,0.5]


STATS_PREFIX = {
    "multicore": "board.processor.cores1.core.",
    "singlecore": "board.processor.cores.core."
}

PDFS = {
    "penalty_lin": "valuePred.penaltyCycles::",
    "savings_lin": "valuePred.valuePredSavedCycles::",
    "penalty_log": "valuePred.penaltyCyclesLog2::",
    "savings_log": "valuePred.valuePredSavedCyclesLog2::",
}

STATS = {
    "ipc": "ipc",
    "coverage": "valuePred.predCoverage",
    "accuracy": "valuePred.accuracy",
    "penalty_total": "valuePred.totalPenaltyCycles",
    "savings_total": "valuePred.totalSavedCycles",
    "pdfs": PDFS
}

data = {
    "script_file": "../scripts/run_important_simpoints.sh",
    "config_file": "../gem5-configs/all-simpoint-run.py",
    "configs_path": "../gem5-configs/conf-configs",
    "stats_file" : "stats.txt",
    "min-confidence" : 2,
    "max-confidence" : 200,
    "confidence" : CONF,
    "save" : SAVE,
    "alpha" : ALPHA, 
    "bms" : BMS,
    "experiment" : "load_data_withstride",
    "stats" : STATS,
    "stats_prefix" : STATS_PREFIX,
    "use_stride" : True
}

data["result_path"]= f"results/{data["experiment"]}/"
data["stats_path"] = f"../scripts/results/arm64/{data["experiment"]}/"


with open("config.json", "w") as f:
    json.dump(data, f, indent=4)

pio.templates.default = "simple_white"


#---CHECKS---#
if data["min-confidence"]>=data["max-confidence"]:
    raise ValueError("Check Confidence values")


#---METHODS----#

def contains_line(path,target_line):
    lines = Path(path).read_text().splitlines()
    if target_line in lines:
        return True
    else:
        return False
    
def contains_regex(path,target_regex):
    lines = Path(path).read_text().splitlines()
    return any(re.search(target_regex,line) for line in lines)

def replace_line(path,target_string,new_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_string):
            lines[i] = new_line
            updated = True
            break
    if not updated:
        raise ValueError(f"Key '{target_string}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")


def insert_line_after(path,target_line,new_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_line):
            lines.insert(i+1,new_line)
            updated = True
            break
    if not updated:
        raise ValueError(f"Key '{target_line}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")

def remove_line(path,target_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_line):
            lines.remove(line)
            updated = True
            break
    if not updated:
        print(f"Key '{target_line}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")




#### 1.2 Setup

In [6]:


# Configure Shell Script (run_important_simpoints.sh)
      
def bms_comments(path):
    target_regex=r'^BMS[+]='
    lines = Path(path).read_text().splitlines()
    for i, line in enumerate(lines):
        if re.search(target_regex,line):
            lines[i]="#"+lines[i]
    Path(path).write_text("\n".join(lines) + "\n") 

    for bm in data["bms"]:
        target_string=f'#BMS+=("{bm}")'
        new_line=f'BMS+=("{bm}")'
        if contains_line(path,new_line):
            print(f"{bm} already in BMS")
        else:
            replace_line(path,target_string,new_line)

def exp_name(path):
    target_string=f"EXPERIMENT="
    new_line=f"{target_string}{data["experiment"]}"
    replace_line(path,target_string,new_line)


def conf_variable(path):
    target_line=f"EXPERIMENT="
    new_line=f"CONF=2"
    target_regex=r"CONF=\s*"
    if contains_regex(path,target_regex):
        print(f"CONF= already in {path}")
        return
    insert_line_after(path,target_line,new_line)


def resdir_setting(path):
    target_string="RESDIR=${RESULTS_DIR}/$bm/"
    new_line=f"\t{target_string}$CONF"
    replace_line(path,target_string,new_line)


# Configure Python Script (all-simpoints-run.py)

def check_valuePred(path):
    target_string = r"cpu.predictValues\s*=\s*True"
    if contains_regex(path,target_string):
            print(f"ValuePredictor currently turned on in {path}")
    else:  
        print(f"\033[1;31mValuePredictor currently not turned on in {path}\033[0m")

    target_string = r"cpu.valuePred.confidence_threshold\s*=\s*"
    if not contains_regex(path,target_string):
        print(f"\033[1;31m cpu.valuePred.confidence_threshold missing in {path}\033[0m")


def add_syspath(path):
    old_line="from pathlib import Path"
    target_string = "import sys"
    if contains_line(path,target_string):
        print(f"{target_string} already in {path}")
    else:
        insert_line_after(path,old_line,target_string)
    old_line=target_string
    target_string = "sys.path.append(str(Path(__file__).resolve().parents[1]))"
    if contains_line(path,target_string):
        print(f"{target_string} already in {path}")
    else:
        insert_line_after(path,old_line,target_string)

def set_stride(path):
    target_line="cpu.valuePred.use_stride ="
    new_line = f"cpu.valuePred.use_stride = {data["use_stride"]}"
    replace_line(path,target_line,new_line)

def config_shell_script():
    path = data["script_file"]
    bms_comments(path)
    exp_name(path)
    conf_variable(path)
    resdir_setting(path) 
    
def config_python_script():
    path = data["config_file"]
    check_valuePred(path)
    add_syspath(path)
    set_stride(path)

#Create Folders
def create_folders():
    path = data["configs_path"]
    Path(path).mkdir(exist_ok=True)


create_folders()
config_shell_script()
config_python_script()



CONF= already in ../scripts/run_important_simpoints.sh
ValuePredictor currently turned on in ../gem5-configs/all-simpoint-run.py
import sys already in ../gem5-configs/all-simpoint-run.py
sys.path.append(str(Path(__file__).resolve().parents[1])) already in ../gem5-configs/all-simpoint-run.py


##### 1.3 Reset Scripts to Default and Cleanup

In [2]:

def bms_uncomments(path):
    target_regex=r'^#BMS[+]='
    lines = Path(path).read_text().splitlines()
    for i, line in enumerate(lines):
        if re.search(target_regex,line):
            lines[i]=lines[i][1:]
    Path(path).write_text("\n".join(lines) + "\n") 

def exp_reset(path):
    target_string=f"EXPERIMENT="
    new_line=f"{target_string}exp"
    replace_line(path,target_string,new_line)


def rm_conf_variable(path):
    target_regex=r"CONF="
    target_line=f"CONF="
    if not contains_regex(path,target_regex):
        print(f"{target_line} already removed from {path}")
        return
    remove_line(path,target_line)

def resdir_setting_reset(path):
    target_string="RESDIR=${RESULTS_DIR}/$bm/$CONF"
    new_line="\tRESDIR=${RESULTS_DIR}/$bm/"
    if contains_line(path,new_line):# and contains_line(path,new_line):
        print(f"{target_string} already replaced in {path}")
        return
    replace_line(path,target_string,new_line)


def reset_config_path(path):
    target_string="CONFIG="
    new_string=f"CONFIG={os.path.abspath(data["config_file"])}"
    if contains_line(path,new_string):
        print(f"{new_string} already replaced in {path}")
    else:
        replace_line(path,target_string,new_string)

# Configure Python Script (all-simpoints-run.py)


def remove_syspath(path):
    target_string = "import sys"
    if not contains_line(path,target_string):
        print(f"{target_string} already removed from {path}")
    else:
        remove_line(path,target_string)
    target_string = "sys.path.append(str(Path(__file__).resolve().parents[1]))"
    if not contains_line(path,target_string):
        print(f"{target_string} already removed from {path}")
    else:
        remove_line(path,target_string)

#------

def reset_python_script():
    path=data["script_file"]
    bms_uncomments(path)
    exp_reset(path)
    resdir_setting_reset(path)
    rm_conf_variable(path)
    reset_config_path(path)
    return

def reset_shell_script():
    path=data["config_file"]
    remove_syspath(path)
    return


def cleanup_folders():
    folder=Path(data["configs_path"])
    if not folder.exists():
        print(f"{folder} folder already deleted")
        return
    shutil.rmtree(folder)
    

cleanup_folders()
reset_python_script()
reset_shell_script()

RESDIR=${RESULTS_DIR}/$bm/$CONF already replaced in ../scripts/run_important_simpoints.sh


## 2. Execution
#### 2.1 Setting Confidence and Starting Executions

In [ ]:


def duplicate_config(new_conf):
    new_config = f"{data["configs_path"]}/all-simpoints-run-{new_conf}.py"
    shutil.copy(data["config_file"],new_config)
    target_line = f"cpu.valuePred.confidence_threshold"
    new_line = f"cpu.valuePred.confidence_threshold = {new_conf}"
    replace_line(new_config,target_line,new_line)
    

def update_script(confidence):
    target_line = f"CONF="
    new_line = f"CONF={confidence}"
    replace_line(data["script_file"],target_line,new_line)
    
    new_config =f"{os.path.abspath(data["configs_path"])}/all-simpoints-run-{confidence}.py"
    target_line = f"CONFIG="
    new_line = f"CONFIG={new_config}"
    replace_line(data["script_file"],target_line,new_line)

def run_program():
    script_dir = os.path.dirname(os.path.abspath(data["script_file"]))
    script_file = os.path.basename(data["script_file"])

    cwd = os.getcwd()
    try:
        os.chdir(script_dir)
        result = subprocess.run(
            f"bash {script_file}",
            shell=True,          
            check=True,
            capture_output=True,
            text=True
        )
        return result.stdout
    finally:
        os.chdir(cwd)

def save_config():
    with open(data["stats_path"]+"config.json", "w") as f:
        json.dump(data, f, indent=4)

def wait_for_program():
    print(f"waiting")
    subprocess.run("pueue wait -g 'arm64-conf'", shell=True, check = True)

data["save"]=data["alpha"]=[]

for confidence in CONF:
    
    print(f"Confidence = {confidence}")

    duplicate_config(confidence)
    update_script(confidence)
    run_program()
save_config()
print("All scripts started")


Confidence = 2


KeyboardInterrupt: 

#### 2.2 Baseline / Run with concrete Confidence

In [7]:
def duplicate_config(conf):
    if conf == -1:
        new_config = f"{data["configs_path"]}/all-simpoints-run-base.py"
        shutil.copy(data["config_file"],new_config)
        target_line = f"cpu.predictValues = "
        new_line = f"cpu.predictValues = False"
        replace_line(new_config,target_line,new_line)
        print("WATCH OUT: Turning off Valuepredictor")
    else:
        new_config = f"{data["configs_path"]}/all-simpoints-run-{conf}.py"
        shutil.copy(data["config_file"],new_config)
        target_line = f"cpu.valuePred.confidence_threshold"
        new_line = f"cpu.valuePred.confidence_threshold = {conf}"
        replace_line(new_config,target_line,new_line)

def update_script(conf):
    resdir_setting_reset(data["script_file"])
    if conf ==-1:
        new_config =f"{os.path.abspath(data["configs_path"])}/all-simpoints-run-base.py"        
    else:
        new_config =f"{os.path.abspath(data["configs_path"])}/all-simpoints-run-{conf}.py"
    target_line = f"CONFIG="
    new_line = f"CONFIG={new_config}"
    replace_line(data["script_file"],target_line,new_line)


def resdir_setting_reset(path):
    target_string="RESDIR=${RESULTS_DIR}/$bm/$CONF"
    new_line="\tRESDIR=${RESULTS_DIR}/$bm/"
    if contains_line(path,new_line):# and contains_line(path,new_line):
        print(f"{target_string} already replaced in {path}")
        return
    replace_line(path,target_string,new_line)

def save_config():
    with open(data["stats_path"]+"config.json", "w") as f:
        json.dump(data, f, indent=4)

def run_program():
    script_dir = os.path.dirname(os.path.abspath(data["script_file"]))
    script_file = os.path.basename(data["script_file"])

    cwd = os.getcwd()
    try:
        os.chdir(script_dir)
        result = subprocess.run(
            f"bash {script_file}",
            shell=True,          
            check=True,
            capture_output=True,
            text=True
        )
        return result.stdout
    finally:
        os.chdir(cwd)

# Input Confidence to run with, -1 for base case aka value Pred turned off:
conf=50

data["save"]=data["alpha"]=[]
data["min-confidence"]=data["max-confidence"]=conf
data["conf"]=[conf]

duplicate_config(conf)
update_script(conf)
run_program()
save_config()


#### 2.3 Saving Threshold Executions

In [ ]:
def duplicate_config(save,conf,alpha):
    new_config = f"{data["configs_path"]}/all-simpoints-run-{save}.py"
    shutil.copy(data["config_file"],new_config)
    target_line = f"cpu.valuePred.savings_threshold"
    new_line = f"cpu.valuePred.savings_threshold = {save}"
    replace_line(new_config,target_line,new_line)

    target_line = f"cpu.valuePred.confidence_threshold"
    new_line = f"cpu.valuePred.confidence_threshold = {conf}"
    replace_line(new_config,target_line,new_line)

    target_line = f"cpu.valuePred.savings_alpha"
    new_line = f"cpu.valuePred.savings_alpha = {alpha}"
    replace_line(new_config,target_line,new_line)

def update_script(save):
    target_line = f"CONF="
    new_line = f"CONF={save}"
    replace_line(data["script_file"],target_line,new_line)
    
    new_config =f"{os.path.abspath(data["configs_path"])}/all-simpoints-run-{save}.py"
    target_line = f"CONFIG="
    new_line = f"CONFIG={new_config}"
    replace_line(data["script_file"],target_line,new_line)

def run_program():
    script_dir = os.path.dirname(os.path.abspath(data["script_file"]))
    script_file = os.path.basename(data["script_file"])

    cwd = os.getcwd()
    try:
        os.chdir(script_dir)
        result = subprocess.run(
            f"bash {script_file}",
            shell=True,          
            check=True,
            capture_output=True,
            text=True
        )
        return result.stdout
    finally:
        os.chdir(cwd)

def save_config():
    with open(data["stats_path"]+"config.json", "w") as f:
        json.dump(data, f, indent=4)

conf=50
data["min-confidence"]=data["max-confidence"]=conf
data["confidence"]=[conf]

alpha=0.1
data["alpha"]=[alpha]


for save in data["save"]:
    
    print(f"Savings Threshold = {save}")

    duplicate_config(save,conf,alpha)
    update_script(save)
    run_program()
save_config()
print("All scripts started")


Savings Threshold = 10
Saving alpha Value = 0.01
Savings Threshold = 10
Saving alpha Value = 0.05
Savings Threshold = 10
Saving alpha Value = 0.1
Savings Threshold = 10
Saving alpha Value = 0.2
Savings Threshold = 10
Saving alpha Value = 0.3
Savings Threshold = 10
Saving alpha Value = 0.4
Savings Threshold = 10
Saving alpha Value = 0.5


FileNotFoundError: [Errno 2] No such file or directory: '../scripts/results/arm64/sp_alpha_variation/config.json'

##### Alpha Variation

In [8]:
def duplicate_config(save,conf,alpha):
    new_config = f"{data["configs_path"]}/all-simpoints-run-{alpha}.py"
    shutil.copy(data["config_file"],new_config)
    target_line = f"cpu.valuePred.savings_threshold"
    new_line = f"cpu.valuePred.savings_threshold = {save}"
    replace_line(new_config,target_line,new_line)

    target_line = f"cpu.valuePred.confidence_threshold"
    new_line = f"cpu.valuePred.confidence_threshold = {conf}"
    replace_line(new_config,target_line,new_line)

    target_line = f"cpu.valuePred.savings_alpha"
    new_line = f"cpu.valuePred.savings_alpha = {alpha}"
    replace_line(new_config,target_line,new_line)

def update_script(alpha):
    target_line = f"CONF="
    new_line = f"CONF={alpha}"
    replace_line(data["script_file"],target_line,new_line)
    
    new_config =f"{os.path.abspath(data["configs_path"])}/all-simpoints-run-{alpha}.py"
    target_line = f"CONFIG="
    new_line = f"CONFIG={new_config}"
    replace_line(data["script_file"],target_line,new_line)

def run_program():
    script_dir = os.path.dirname(os.path.abspath(data["script_file"]))
    script_file = os.path.basename(data["script_file"])

    cwd = os.getcwd()
    try:
        os.chdir(script_dir)
        result = subprocess.run(
            f"bash {script_file}",
            shell=True,          
            check=True,
            capture_output=True,
            text=True
        )
        return result.stdout
    finally:
        os.chdir(cwd)

def save_config():
    with open(data["stats_path"]+"config.json", "w") as f:
        json.dump(data, f, indent=4)

conf=50
data["min-confidence"]=data["max-confidence"]=conf
data["confidence"]=[conf]

save=10
data["save"]=[save]

for alpha in data["alpha"]:
    
    print(f"Saving alpha Value = {alpha}")

    duplicate_config(save,conf,alpha)
    update_script(alpha)
    run_program()
save_config()
print("All scripts started")


Saving alpha Value = 0.01
Saving alpha Value = 0.05
Saving alpha Value = 0.1
Saving alpha Value = 0.2
Saving alpha Value = 0.3
Saving alpha Value = 0.4
Saving alpha Value = 0.5
All scripts started


## 4. Complete Cleanup
#### 4.1 CSV Results

In [3]:
path = Path(data["result_path"])
if path.exists():
    shutil.rmtree(path)

#### 4.2 Simulation Results (USE CAREFULLY)

In [12]:
#shutil
path = Path(data["stats_path"])
if path.exists():
    confirm = input("Confirm deleting of simulation Results? (yes/no): ")
    if confirm =="yes":
        shutil.rmtree(path)
        print("Results deleted")
    else:
        print("Cancelled")

Cancelled
